# Charts for thesis

## Classification

### Confusion matrix

In [ ]:
from pyexpat import model

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_curve, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Load predictions using pathlib
preds_path = Path("test_results") / "Classification" / "predictions" / "test_preds_100k_tuned.parquet"
df_preds = pd.read_parquet(preds_path)

model_name = "LightGBM"
y_true = df_preds["true_y"]
y_pred_proba = df_preds[model_name]

# Optimal threshold calculation (Youden's J statistic)
fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

# Class predictions using the optimal threshold
y_pred_class = (y_pred_proba >= optimal_threshold).astype(int)

# Confusion matrix elements
tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()

# Dynamically calculate cell colors using matplotlib's "Blues" colormap
cmap = plt.get_cmap("Blues")
vmax = max(tn, fp, fn, tp)

# Multiply vmax to ensure the darkest blue is light enough for clean black text
norm = mcolors.Normalize(vmin=0, vmax=vmax * 2.6)

def get_cell_color(val):
    # Extracts the hex code without the alpha channel
    return mcolors.to_hex(cmap(norm(val)))[1:].upper()

hex_tn, hex_fp, hex_fn, hex_tp = map(get_cell_color, [tn, fp, fn, tp])

# Generate the native LaTeX TikZ code with a clean, modern aesthetic
latex_code = f"""
\\begin{{figure}}[htbp]
\\centering
\\begin{{tikzpicture}}[
    cell/.style={{rectangle, minimum width=3.2cm, minimum height=3.2cm, align=center, font=\\sffamily, text=black}},
    axisl/.style={{font=\\sffamily\\bfseries}}
]
    \\definecolor{{colorTN}}{{HTML}}{{{hex_tn}}}
    \\definecolor{{colorFP}}{{HTML}}{{{hex_fp}}}
    \\definecolor{{colorFN}}{{HTML}}{{{hex_fn}}}
    \\definecolor{{colorTP}}{{HTML}}{{{hex_tp}}}

    \\node[cell, fill=colorTN] (TN) at (0, 0) {{\\Large TN\\\\[0.2cm] \\large {tn}}};
    \\node[cell, fill=colorFP] (FP) at (3.3, 0) {{\\Large FP\\\\[0.2cm] \\large {fp}}};
    \\node[cell, fill=colorFN] (FN) at (0, -3.3) {{\\Large FN\\\\[0.2cm] \\large {fn}}};
    \\node[cell, fill=colorTP] (TP) at (3.3, -3.3) {{\\Large TP\\\\[0.2cm] \\large {tp}}};

    \\node[axisl] at (1.65, 2.3) {{Predikovaná třída}};
    \\node[axisl] at (0, 1.8) {{0}};
    \\node[axisl] at (3.3, 1.8) {{1}};
    
    \\node[axisl, rotate=90] at (-2.3, -1.65) {{Skutečná třída}};
    \\node[axisl] at (-1.8, 0) {{0}};
    \\node[axisl] at (-1.8, -3.3) {{1}};

\\end{{tikzpicture}}
\\caption{{Matice záměn -- {model_name} (100k dataset). Optimální hranice: {optimal_threshold:.4f}.}}
\\label{{fig:matice_zamen_{model_name.lower()}_100k}}
\\end{{figure}}
"""

print(latex_code)


\begin{figure}[htbp]
\centering
\begin{tikzpicture}[
    cell/.style={rectangle, minimum width=3.2cm, minimum height=3.2cm, align=center, font=\sffamily, text=black},
    axisl/.style={font=\sffamily\bfseries}
]
    \definecolor{colorTN}{HTML}{9AC8E0}
    \definecolor{colorFP}{HTML}{CDDFF1}
    \definecolor{colorFN}{HTML}{EEF5FC}
    \definecolor{colorTP}{HTML}{E0ECF8}

    \node[cell, fill=colorTN] (TN) at (0, 0) {\Large TN\\[0.2cm] \large 135055};
    \node[cell, fill=colorFP] (FP) at (3.3, 0) {\Large FP\\[0.2cm] \large 76003};
    \node[cell, fill=colorFN] (FN) at (0, -3.3) {\Large FN\\[0.2cm] \large 17495};
    \node[cell, fill=colorTP] (TP) at (3.3, -3.3) {\Large TP\\[0.2cm] \large 40055};

    \node[axisl] at (1.65, 2.3) {Predikovaná třída};
    \node[axisl] at (0, 1.8) {0};
    \node[axisl] at (3.3, 1.8) {1};
    
    \node[axisl, rotate=90] at (-2.3, -1.65) {Skutečná třída};
    \node[axisl] at (-1.8, 0) {0};
    \node[axisl] at (-1.8, -3.3) {1};

\end{tikzpicture}
\caption{Ma

### ROC Curve

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score

# Load data and probabilities for models from the 100k tuned dataset
preds_path = Path("test_results") / "Classification" / "predictions" / "test_preds_100k_tuned.parquet"
df_preds = pd.read_parquet(preds_path)

y_true = df_preds["true_y"]
model_name = "LightGBM"
y_pred_proba = df_preds[model_name]

# Calculate ROC curve and AUC score for LightGBM
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
auc_score = roc_auc_score(y_true, y_pred_proba)

# Downsample points to approx ~200 to prevent pdflatex memory limits
step = max(1, len(fpr) // 200)
fpr_reduced = fpr[::step]
tpr_reduced = tpr[::step]

# Ensure the curve naturally ends at exactly (1, 1)
if fpr_reduced[-1] != 1.0 or tpr_reduced[-1] != 1.0:
    fpr_reduced = np.append(fpr_reduced, 1.0)
    tpr_reduced = np.append(tpr_reduced, 1.0)
    
coords = "\n        ".join([f"({f:.4f}, {t:.4f})" for f, t in zip(fpr_reduced, tpr_reduced)])

# Generate the native LaTeX PGFPlots code
latex = f"""\\begin{{figure}}[htbp]
\\centering
\\begin{{tikzpicture}}
\\begin{{axis}}[
    width=8cm,
    height=8cm,
    scale only axis=true,
    title={{\\textbf{{ROC křivka -- {model_name} (AUC = {auc_score:.4f})}}}},
    xlabel={{False Positive Rate}},
    ylabel={{True Positive Rate}},
    xmin=-0.02, xmax=1.02,
    ymin=-0.02, ymax=1.02,
    grid=major,
    grid style={{dashed, gray!30}},
    enlargelimits=false,
    tick label style={{font=\\sffamily}},
    label style={{font=\\sffamily\\bfseries}},
    title style={{font=\\sffamily\\bfseries}}
]

    \\addplot [
        color=blue,
        thick,
        solid
    ] coordinates {{
        {coords}
    }};

\\end{{axis}}
\\end{{tikzpicture}}
\\caption{{ROC křivka modelu {model_name} trénovaného na datové sadě o 100 tisících vzorcích, vyhodnoceno na testovací množině.}}
\\label{{fig:roc_auc_100k_lgbm}}
\\end{{figure}}
"""

print(latex)

\begin{figure}[htbp]
\centering
\begin{tikzpicture}
\begin{axis}[
    width=8cm,
    height=8cm,
    scale only axis=true,
    title={\textbf{ROC křivka -- LightGBM (AUC = 0.7315)}},
    xlabel={False Positive Rate},
    ylabel={True Positive Rate},
    xmin=-0.02, xmax=1.02,
    ymin=-0.02, ymax=1.02,
    grid=major,
    grid style={dashed, gray!30},
    enlargelimits=false,
    tick label style={font=\sffamily},
    label style={font=\sffamily\bfseries},
    title style={font=\sffamily\bfseries}
]

    \addplot [
        color=blue,
        thick,
        solid
    ] coordinates {
        (0.0000, 0.0000)
        (0.0014, 0.0113)
        (0.0028, 0.0213)
        (0.0043, 0.0304)
        (0.0059, 0.0387)
        (0.0076, 0.0474)
        (0.0093, 0.0547)
        (0.0110, 0.0634)
        (0.0128, 0.0704)
        (0.0145, 0.0775)
        (0.0163, 0.0851)
        (0.0181, 0.0916)
        (0.0199, 0.0994)
        (0.0217, 0.1065)
        (0.0236, 0.1135)
        (0.0254, 0.1205)
        (0

### Results

In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths to the score directories
scores_dir = Path("test_results") / "Classification" / "scores"
datasets = ["1k", "10k", "100k", "full"]

# Data structure [dataset][model_base]
roc_auc_data = {ds: {} for ds in datasets}
all_models = set()

# Load data into dictionaries
for ds in datasets:
    file = scores_dir / f"test_scores_{ds}_tuned.csv"
    if not file.exists():
        continue
        
    df = pd.read_csv(file)
    for _, row in df.iterrows():
        model_raw = row["Model"]
        
        # Exclude Dummy model entirely
        if "Dummy" in model_raw:
            continue
        
        # Determine if the model was trained on a smaller dataset or not tuned
        has_asterisk = "(not_tuned)" in model_raw or "(10k)" in model_raw or "(100k)" in model_raw or "(30k)" in model_raw
        model_base = model_raw.split(" (")[0]
        
        all_models.add(model_base)
        
        # Store as a tuple: (value, boolean_has_asterisk)
        if has_asterisk:
            roc_auc_data[ds][model_base] = (None, True)
        else:
            roc_auc_data[ds][model_base] = (row["ROC AUC"], False)

# Sort models alphabetically to ensure visual consistency
all_models = sorted(list(all_models))

# Function to generate the transposed LaTeX table with booktabs formatting
def generate_latex_table_transposed(data_dict, title, label):
    # Příprava hlaviček
    header_models = " & ".join([f"\\textbf{{{m}}}" for m in all_models])
    header_empty = " & ".join(["" for _ in all_models])
    
    latex = f"""\\begin{{table}}[htbp]
\\centering
% -- Zúžení tabulky --
\\small
\\setlength{{\\tabcolsep}}{{4.5pt}}
\\renewcommand{{\\arraystretch}}{{1.25}}
% --------------------
\\begin{{tabular}}{{l {' c' * len(all_models)}}}
\\toprule
\\textbf{{Počet záznamů}} & {header_models} \\\\[-0.3em]
\\textbf{{trénovací sady}} & {header_empty} \\\\
\\midrule
"""
    
    # Najdeme nejlepší model pro každou datovou sadu, abychom ho mohli zvýraznit
    best_per_ds = {}
    for ds in datasets:
        valid_vals = []
        for m in all_models:
            if m in data_dict[ds]:
                val, is_ast = data_dict[ds][m]
                if not is_ast and val is not None:
                    valid_vals.append(val)
        
        if valid_vals:
            best_per_ds[ds] = max(valid_vals)  # Pro ROC AUC hledáme maximum
        else:
            best_per_ds[ds] = None

    has_global_asterisk = False
    
    # Generování řádků přes datové sady
    for ds in datasets:
        row_str = f"\\textbf{{{ds}}} "
        for model in all_models:
            if model in data_dict[ds]:
                val, is_ast = data_dict[ds][model]
                
                if is_ast:
                    has_global_asterisk = True
                    row_str += "& $^*$ "
                elif val is not None:
                    # Česká desetinná čárka zapouzdřená do závorek pro LaTeX
                    num_str = f"{val:.4f}".replace(".", "{,}")
                    
                    # Zvýraznění nejlepší buňky (na úrovni dané datové sady)
                    if best_per_ds[ds] is not None and abs(val - best_per_ds[ds]) < 1e-6:
                        row_str += f"& \\cellcolor{{green!25}}\\textbf{{{num_str}}} "
                    else:
                        row_str += f"& {num_str} "
                else:
                    row_str += "& - "
            else:
                row_str += "& - "
                
        row_str += "\\\\\n"
        latex += row_str
        
    latex += "\\bottomrule\n\\end{tabular}\n"
    
    # Poznámka pod čarou, pokud existují chybějící hodnoty
    if has_global_asterisk:
        footnote_msg = "U těchto modelů nebylo možné z časových důvodů provést ladění hyperparametrů na plné datové sadě (GBM, NGBoost), nebo model neumožňuje trénování na takto velkých datech (TabPFN)."
        latex += f"\\\\ {{\\footnotesize $^*$ {footnote_msg}}}\n"
        
    latex += f"\\caption{{{title}}}\n\\label{{{label}}}\n\\end{{table}}\n"
    return latex

print("% -- TRANSPOSED ROC AUC TABLE --")
print(generate_latex_table_transposed(
    roc_auc_data, 
    "ROC AUC skóre modelů napříč trénovacími sadami vyhodnocené na stejné testovací množině.", 
    "tab:roc_auc_scores"
))

% -- TRANSPOSED ROC AUC TABLE --
\begin{table}[htbp]
\centering
% -- Zúžení tabulky --
\small
\setlength{\tabcolsep}{4.5pt}
\renewcommand{\arraystretch}{1.25}
% --------------------
\begin{tabular}{l  c c c c c c c}
\toprule
\textbf{Počet záznamů} & \textbf{CatBoost} & \textbf{GBM} & \textbf{HistGBM} & \textbf{LightGBM} & \textbf{NGBoost} & \textbf{TabPFN} & \textbf{XGBoost} \\[-0.3em]
\textbf{trénovací sady} &  &  &  &  &  &  &  \\
\midrule
\textbf{1k} & 0{,}6794 & 0{,}6853 & 0{,}6767 & 0{,}6837 & 0{,}6806 & \cellcolor{green!25}\textbf{0{,}6984} & 0{,}6688 \\
\textbf{10k} & 0{,}7062 & 0{,}7109 & 0{,}7102 & 0{,}7137 & 0{,}7118 & \cellcolor{green!25}\textbf{0{,}7177} & 0{,}7117 \\
\textbf{100k} & 0{,}7298 & 0{,}7223 & 0{,}7284 & \cellcolor{green!25}\textbf{0{,}7315} & 0{,}7304 & 0{,}7302 & 0{,}7305 \\
\textbf{full} & 0{,}7260 & $^*$ & 0{,}7272 & 0{,}7286 & $^*$ & $^*$ & \cellcolor{green!25}\textbf{0{,}7300} \\
\bottomrule
\end{tabular}
\\ {\footnotesize $^*$ U těchto modelů nebylo možné

## Regression

### Results

In [37]:
import pandas as pd
import numpy as np
from pathlib import Path

# Cesty ke složkám s výsledky regrese
scores_dir = Path("test_results") / "Regression" / "scores"
datasets = ["1k", "10k", "100k", "full"]

# Datové struktury
rmse_data = {ds: {} for ds in datasets}
all_models = set()

# Načtení dat
for ds in datasets:
    file = scores_dir / f"test_scores_{ds}_tuned.csv"
    if not file.exists():
        continue
        
    df = pd.read_csv(file)
    for _, row in df.iterrows():
        model_raw = row["Model"]
        
        # Kompletní ignorování Dummy modelu
        if "Dummy" in model_raw:
            continue
        
        has_asterisk = "(not_tuned)" in model_raw or "(10k)" in model_raw or "(100k)" in model_raw or "(30k)" in model_raw
        model_base = model_raw.split(" (")[0]
        
        all_models.add(model_base)
        
        if has_asterisk:
            rmse_data[ds][model_base] = (None, True)
        else:
            rmse_data[ds][model_base] = (row["RMSE"], False)

all_models = sorted(list(all_models))

def generate_latex_table_rmse_transposed(data_dict, title, label):
    # Příprava hlaviček
    header_models = " & ".join([f"\\textbf{{{m}}}" for m in all_models])
    header_empty = " & ".join(["" for _ in all_models])
    
    latex = f"""\\begin{{table}}[htbp]
\\centering
\\resizebox{{\\textwidth}}{{!}}{{
\\small
\\setlength{{\\tabcolsep}}{{3.5pt}}
\\renewcommand{{\\arraystretch}}{{1.25}}
\\begin{{tabular}}{{l {' c' * len(all_models)}}}
\\toprule
\\textbf{{Počet záznamů}} & {header_models} \\\\[-0.3em]
\\textbf{{trénovací sady}} & {header_empty} \\\\
\\midrule
"""
    
    # U RMSE hledáme MINIMUM (nejmenší chybu)
    best_per_ds = {}
    for ds in datasets:
        valid_vals = []
        for m in all_models:
            if m in data_dict[ds]:
                val, is_ast = data_dict[ds][m]
                if not is_ast and val is not None:
                    valid_vals.append(val)
        
        if valid_vals:
            best_per_ds[ds] = min(valid_vals) # Pro RMSE hledáme minimum
        else:
            best_per_ds[ds] = None

    has_global_asterisk = False
    
    # Generování řádků přes datové sady
    for ds in datasets:
        row_str = f"\\textbf{{{ds}}} "
        for model in all_models:
            if model in data_dict[ds]:
                val, is_ast = data_dict[ds][model]
                
                if is_ast:
                    has_global_asterisk = True
                    row_str += "& $^*$ "
                elif val is not None:
                    # Česká desetinná čárka zapouzdřená do závorek pro LaTeX
                    num_str = f"{val:.4f}".replace(".", "{,}")
                    
                    # Zvýraznění nejlepší buňky (na úrovni dané datové sady)
                    if best_per_ds[ds] is not None and abs(val - best_per_ds[ds]) < 1e-6:
                        row_str += f"& \\cellcolor{{green!25}}\\textbf{{{num_str}}} "
                    else:
                        row_str += f"& {num_str} "
                else:
                    row_str += "& - "
            else:
                row_str += "& - "
                
        row_str += "\\\\\n"
        latex += row_str
        
    latex += "\\bottomrule\n\\end{tabular}\n"
    latex += "} % Ukonceni resizeboxu\n"
    
    # Poznámka pod čarou
    if has_global_asterisk:
        footnote_msg = "U těchto modelů nebylo proveditelné z časových důvodů provést ladění nebo trénování na plné sadě (GBM, NGBoost, TabPFN, PGBM)."
        latex += f"\\vspace{{0.1cm}}\\\\ {{\\footnotesize $^*$ {footnote_msg}}}\n"
        
    latex += f"\\caption{{{title}}}\n\\label{{{label}}}\n\\end{{table}}\n"
    return latex

print("% -- TRANSPOSED RMSE TABLE --")
print(generate_latex_table_rmse_transposed(
    rmse_data, 
    "RMSE skóre regresních modelů napříč trénovacími sadami vyhodnocené na stejné testovací množině.", 
    "tab:rmse_scores"
))

% -- TRANSPOSED RMSE TABLE --
\begin{table}[htbp]
\centering
\resizebox{\textwidth}{!}{
\small
\setlength{\tabcolsep}{3.5pt}
\renewcommand{\arraystretch}{1.25}
\begin{tabular}{l  c c c c c c c c}
\toprule
\textbf{Počet záznamů} & \textbf{CatBoost} & \textbf{GBM} & \textbf{HistGBM} & \textbf{LightGBM} & \textbf{NGBoost} & \textbf{PGBM} & \textbf{TabPFN} & \textbf{XGBoost} \\[-0.3em]
\textbf{trénovací sady} &  &  &  &  &  &  &  &  \\
\midrule
\textbf{1k} & 0{,}3613 & 0{,}3631 & 0{,}3628 & 0{,}3622 & 0{,}3624 & 0{,}3637 & \cellcolor{green!25}\textbf{0{,}3609} & 0{,}3620 \\
\textbf{10k} & 0{,}3590 & 0{,}3601 & 0{,}3602 & 0{,}3589 & \cellcolor{green!25}\textbf{0{,}3583} & 0{,}3602 & 0{,}3589 & 0{,}3586 \\
\textbf{100k} & 0{,}3533 & 0{,}3541 & 0{,}3542 & 0{,}3533 & 0{,}3544 & 0{,}3541 & - & \cellcolor{green!25}\textbf{0{,}3531} \\
\textbf{full} & 0{,}3536 & $^*$ & 0{,}3539 & \cellcolor{green!25}\textbf{0{,}3534} & $^*$ & 0{,}3542 & - & 0{,}3537 \\
\bottomrule
\end{tabular}
} % Ukonceni resiz

### PGBM and NGBoost